In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from statsmodels.sandbox.tools.cross_val import LeaveOneOut

In [ ]:
#df_arduino = pd.read_csv("../ArduinoApps/emg_auswertung/tsfel_debug.csv", index_col=0)
df_full = pd.read_csv("../auswertung_features/cache/feature_trim=0-minlen=0-win=0-step=50-fs=default_tsfel.csv", index_col=0)
df_expect = df_full.iloc[:,6:]

df_lorenzen = pd.read_csv("../auswertung_features/cache/feature_trim=0-minlen=0-win=0-step=50-fs=tsfel.csv", index_col=0)
a = df_lorenzen.filter(regex="^0_*", axis=1)
print(df_lorenzen.iloc[:,6:].shape, a.shape)

In [ ]:
# df_a: "normale" Spaltennamen
# df_b: hat ggf. "sensor_" als Präfix

# 1) Spalten von df_b normalisieren (nur wenn sie mit "sensor_" anfangen)
df_b_norm_cols = df_arduino.columns.str.replace(r"^sensor_", "", regex=True)

# Optional: Kopie mit umbenannten Spalten erstellen
df_b_norm = df_arduino.copy()
df_b_norm.columns = df_b_norm_cols

# 2) Vergleich: welche Spalten fehlen wo?
cols_a = set(df_expect.columns)
cols_b = set(df_b_norm.columns)

b = list(df_b_norm.columns)
count = 0
for i, column in  enumerate(df_expect.columns):
       if column != b[i]:
           print(column)
       else:
           count += 1
print(f"Anzahl an gleichen headern: {count}")
fehlen_in_b = sorted(cols_a - cols_b)  # in df_a vorhanden, aber nicht in df_b (nach Normalisierung)
fehlen_in_a = sorted(cols_b - cols_a)  # in df_b vorhanden, aber nicht in df_a

print("Fehlen in df_b:", fehlen_in_b)
print("Anzahl:", len(fehlen_in_b))
print("\n\nFehlen in df_a:", fehlen_in_a)
print("Anzahl:", len(fehlen_in_a))

# 3) (optional) Gemeinsame Spalten
gemeinsam = sorted(cols_a & cols_b)
print("\n\nGemeinsam:", gemeinsam)


# Test Model on Input again

In [ ]:
import pandas as pd
import numpy as np
from auswertung_features.build_features import split_into_label_blocks, load_session
from ArduinoApps.emg_auswertung.python.features import building_feature, block_to_win
from ANN.loso_windowed_blockfeat import compute_W_max, block_to_windows_postpad

# load dataset
session = load_session("../ArduinoApps/emg_messung/python/messdaten/messung_r_2.csv")
blocks = split_into_label_blocks(session)

test_block = blocks[1][0] # taking the 0 block

feats_df = building_feature(pd.DataFrame(test_block))
win = block_to_win(test_block, T = 500, stride = 250)

W_max = compute_W_max(2000, 500, 250)
X_ts_block, wlen, mask_block = block_to_win(test_block, T = 500, stride = 250, W_max=W_max)

In [ ]:
import tensorflow as tf
import pickle

tf_model = tf.keras.models.load_model("models/EMG-CNN-Model.keras")
scaler = pickle.load(open("models/scaler_features.pkl", "rb"))

X_feat = np.asarray(feats_df, dtype=np.float32)
X_feat = scaler.transform(X_feat).astype(np.float32)
X_feat3 = np.repeat(X_feat, repeats=win.shape[0], axis=0)  # (3, 624)

prob = tf_model.predict({"ts": win, "feat": X_feat3}, verbose=0)
y_pred = np.argmax(prob, axis=1)
print(y_pred)

In [ ]:
from ai_edge_litert.interpreter import Interpreter

lite_scaler =  pickle.load(open("models/scaler_features.pkl", "rb"))
tflite_model = Interpreter(model_path="models/EMG-CNN.tflite")
tflite_model.allocate_tensors()

in_details = tflite_model.get_input_details()
out_details = tflite_model.get_output_details()

N = win.shape[0]
X_feat = np.asarray(feats_df, dtype=np.float32)
X_feat = scaler.transform(X_feat).astype(np.float32)
#X_feat3 = np.repeat(X_feat, repeats=win.shape[0], axis=0)  # (3, 624)

probs = np.zeros((N, 5), dtype=np.float32)

for i in range(N):
    x_ts = win[i:i+1, :, :].astype(np.float32)       # (1,500,4)

    tflite_model.set_tensor(in_details[0]["index"], X_feat)    # (1,624)
    tflite_model.set_tensor(in_details[1]["index"], x_ts)      # (1,500,4)
    tflite_model.invoke()

    probs[i] = tflite_model.get_tensor(out_details[0]["index"])[0]  # (5,)

prob_block = probs.mean(axis=0)
print(int(np.argmax(prob_block)))

In [ ]:
import tensorflow as tf
import pickle

gru_model = tf.keras.models.load_model("models/EMG-GRU-Model.keras")
scaler_gru = pickle.load(open("models/scaler_features_GRU.pkl", "rb"))

X_feat = np.asarray(feats_df, dtype=np.float32)
X_feat = scaler_gru.transform(X_feat).astype(np.float32)

prob = gru_model.predict({"ts": X_ts_block, "feat": X_feat, "mask":mask_block}, verbose=0)
print(prob)
pred  = int(np.argmax(prob[0]))
print(pred)

# Testing Testing Testing

In [1]:
import pandas as pd
import numpy as np
from auswertung_features.build_features import split_into_label_blocks, load_session
from sklearn.model_selection import LeaveOneGroupOut

meta = pd.read_csv("../auswertung_features/meta.csv")
dict_blocks = {}
meta_blocks = {"subject_id":[], "hand": [], "rel_path":[], "block_id":[]}
for i, r in meta.iterrows():
    csv_path = r["rel_path"]
    df_session = load_session(csv_path)

    #Error Handling für leere dfs
    if df_session is None:
        continue

    blocks = split_into_label_blocks(df_session)
    for block in blocks:
        meta_blocks["subject_id"].append(r["subject_id"])
        meta_blocks["hand"].append(r["hand"])
        meta_blocks["rel_path"].append(r["rel_path"])
        meta_blocks["block_id"].append(len(dict_blocks))
        dict_blocks[len(dict_blocks)] = block
meta_blocks = pd.DataFrame(meta_blocks)
meta_blocks_r = meta_blocks[meta_blocks["hand"]=="r"].reset_index(drop=True)
meta_blocks_l = meta_blocks[meta_blocks["hand"]=="l"].reset_index(drop=True)

/home/manhl/anaconda3/envs/bpa-ba/lib/python3.13/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
I0000 00:00:1788371041.209963  297451 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788371041.210398  297451 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788371041.240489  297451 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788371042.147380  297451 port.cc:

[WARN] ../ArduinoApps/emg_messung/python/messdaten/messung_l_170.csv: is empty


In [2]:
from loso_windowed_blockfeat import block_to_windows_postpad
import tsfel
import os
feature_cache: dict[str, pd.DataFrame] = {}
def tsfel_feature_read(dict_blocks, tsfel_cfg, feature_set_name = "default_tsfel", fs = 500):
    k = feature_set_name
    feature_list = []
    if k not in feature_cache:
        if os.path.isfile(f"../auswertung_features/cache/feature_{k}.csv"):
            feature_cache[k] = pd.read_csv(f"../auswertung_features/cache/feature_{k}.csv", index_col=0)
        else:
            for win, label in dict_blocks.values():
                X = block_to_windows_postpad(win, T=2000, stride=2000)
                feats_df= tsfel.time_series_features_extractor(tsfel_cfg, X, fs=fs)
                feature_list.append(feats_df.iloc[0].to_dict())
            feature_cache[k] = pd.DataFrame(feature_list)
            feature_cache[k].to_csv(f"../auswertung_features/cache/feature_{k}.csv")
    return feature_cache[k]

In [5]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
def logo_baseline(model, meta_block, dict_blocks, tsfel_cfg, feature_set_name= "default_tsfel"):
    logo = LeaveOneGroupOut()
    groups = meta_block["subject_id"].to_numpy()
    fold_metrics = []
    feat_global = tsfel_feature_read(dict_blocks, tsfel_cfg, feature_set_name= feature_set_name)
    for fold, (train_idx, val_idx) in enumerate(logo.split(meta_block, groups=groups)):
        y_list_tr = []
        y_list_va = []
        meta_tr = meta_block.iloc[train_idx].reset_index(drop=True)
        meta_va = meta_block.iloc[val_idx].reset_index(drop=True)

        #Xf_va = feat_global.loc[meta_va["block_id"].astype(int), feat_cols].to_numpy()
        for i, bid in enumerate(meta_tr["block_id"].to_numpy()):
            x_ts_raw, y_i = dict_blocks[int(bid)]
            y_list_tr.append(int(y_i))

        for i, bid in enumerate(meta_va["block_id"].to_numpy()):
            x_ts_raw, y_i = dict_blocks[int(bid)]
            y_list_va.append(int(y_i))

        y_tr = np.array(y_list_tr, dtype=np.int32)
        y_va = np.array(y_list_va, dtype=np.int32)

        X_all = feat_global
        X_tr = X_all.iloc[train_idx].reset_index(drop=True).to_numpy(np.float32)
        X_va = X_all.iloc[val_idx].reset_index(drop=True).to_numpy(np.float32)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va)

        # Metrics
        acc = accuracy_score(y_va, y_pred)
        f1m = f1_score(y_va, y_pred, average="macro")
        cm = confusion_matrix(y_va, y_pred)

        # subject id in this fold is single subject
        subject_left_out = meta_va["subject_id"].unique()
        subject_left_out = int(subject_left_out[0]) if len(subject_left_out) == 1 else subject_left_out.tolist()

        fold_metrics.append({
            "fold": fold,
            "left_out_subject": subject_left_out,
            "n_val": len(y_va),
            "val_acc": float(acc),
            "val_macro_f1": float(f1m),
            "cm_pro_": cm,
        })

    return fold_metrics

In [57]:
def logo_mlp(model, meta_block, dict_blocks, tsfel_cfg, feature_set_name = "default_tsfel", b_CNN = False):
    logo = LeaveOneGroupOut()
    groups = meta_block["subject_id"].to_numpy()
    fold_metrics = []
    feat_global = tsfel_feature_read(dict_blocks, tsfel_cfg, feature_set_name= feature_set_name)

    for fold, (train_idx, val_idx) in enumerate(logo.split(meta_block, groups=groups)):
        y_list_tr = []
        y_list_va = []
        xts_list_tr = []
        xts_list_va = []
        meta_tr = meta_block.iloc[train_idx].reset_index(drop=True)
        meta_va = meta_block.iloc[val_idx].reset_index(drop=True)

        #Xf_va = feat_global.loc[meta_va["block_id"].astype(int), feat_cols].to_numpy()
        for i, bid in enumerate(meta_tr["block_id"].to_numpy()):
            x_tr_raw, y_i = dict_blocks[int(bid)]
            y_list_tr.append(int(y_i))
            xts_list_tr.append(block_to_windows_postpad(x_tr_raw, T = 2000, stride=2000)[0])

        for i, bid in enumerate(meta_va["block_id"].to_numpy()):
            x_va_raw, y_i = dict_blocks[int(bid)]
            y_list_va.append(int(y_i))
            xts_list_va.append(block_to_windows_postpad(x_va_raw, T = 2000, stride=2000)[0])

        y_tr = np.array(y_list_tr, dtype=np.int32)
        y_va = np.array(y_list_va, dtype=np.int32)

        X_all = feat_global
        X_tr = X_all.iloc[train_idx].reset_index(drop=True).to_numpy(np.float32)
        X_va = X_all.iloc[val_idx].reset_index(drop=True).to_numpy(np.float32)

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_va = scaler.transform(X_va)

        es = tf.keras.callbacks.EarlyStopping(
            monitor="loss", patience=10, restore_best_weights=True
        )
        rlr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="loss", factor=0.5, patience=max(2, 10 // 2), min_lr=1e-6
        )
        if b_CNN:
            xts_tr = np.stack(xts_list_tr, axis=0).astype(np.float32)
            xts_va = np.stack(xts_list_va, axis=0).astype(np.float32)
            hist = model.fit({"ts":xts_tr, "feat":X_tr}, y_tr, batch_size=32, epochs=200, callbacks=[es, rlr], shuffle=True)
            y_prob = model.predict({"ts":xts_va, "feat":X_va})
        else:
            hist = model.fit(X_tr, y_tr,
                         validation_data=(X_va, y_va), callbacks=[es, rlr],
                         epochs = 200, shuffle = True, batch_size=32)
            y_prob = model.predict(X_va)

        y_pred = y_prob.argmax(axis=1)

        # Metrics
        acc = accuracy_score(y_va, y_pred)
        f1m = f1_score(y_va, y_pred, average="macro")
        cm = confusion_matrix(y_va, y_pred)

        # subject id in this fold is single subject
        subject_left_out = meta_va["subject_id"].unique()
        subject_left_out = int(subject_left_out[0]) if len(subject_left_out) == 1 else subject_left_out.tolist()

        fold_metrics.append({
            "fold": fold,
            "left_out_subject": subject_left_out,
            "n_val_blocks": int(len(meta_va)),
            "best_epoch": int(np.argmin(hist.history["loss"]) + 1) if "loss" in hist.history else None,
            "n_val": len(y_va),
            "val_acc": float(acc),
            "val_macro_f1": float(f1m),
            "cm_pro_": cm,
        })

    return fold_metrics

## Models to test

In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
standard = StandardScaler()
svc = SVC()
linear_svc = make_pipeline(standard, svc)
svc_one_vs_one = make_pipeline(standard, svc)

feature_set = "default_tsfel"

if feature_set == "default_tsfel":
    tsfel_cfg = tsfel.get_features_by_domain()
else:
    tsfel_cfg = tsfel.get_features_by_domain(json_path="../auswertung_features/tsfel_conf.json")

results_svc = logo_baseline(linear_svc, meta_blocks, dict_blocks, feature_set_name = feature_set, tsfel_cfg = tsfel_cfg)

pd.DataFrame(results_svc).to_csv(f"../auswertung_features/results/logo/results_logo_linear_svc_{feature_set}.csv")

In [17]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

feature_set = "tsfel"

if feature_set == "tsfel":
    tsfel_cfg = tsfel.get_features_by_domain()
else:
    tsfel_cfg = tsfel.get_features_by_domain(json_path="../auswertung_features/tsfel_conf.json")

results_rf = logo_baseline(rf, meta_blocks, dict_blocks, feature_set_name = feature_set, tsfel_cfg = tsfel_cfg)
pd.DataFrame(results_rf).to_csv(f"../auswertung_features/results/logo/results_logo_rf_{feature_set}.csv")

In [61]:
from LOSO_Run_2_input import build_model_feat, build_model
import tensorflow as tf

feature_set = "tsfel"

if feature_set == "default_tsfel":
    tsfel_cfg = tsfel.get_features_by_domain()
    F = 624
else:
    tsfel_cfg = tsfel.get_features_by_domain(json_path="../auswertung_features/tsfel_conf.json")
    F = 260
#tf_model = build_model(2000,F)
#results_mlp = logo_mlp(tf_model, meta_blocks ,dict_blocks, feature_set_name = feature_set, tsfel_cfg = tsfel_cfg, b_CNN=True)
tf_model = build_model_feat(F)
results_mlp = logo_mlp(tf_model, meta_blocks, dict_blocks, feature_set_name = feature_set,tsfel_cfg = tsfel_cfg)
pd.DataFrame(results_mlp).to_csv(f"../auswertung_features/results/logo/results_logo_mlp_{feature_set}.csv")

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2462 - ce: 2.5752 - loss: 3.2071 - val_accuracy: 0.4667 - val_ce: 1.1228 - val_loss: 1.7496 - learning_rate: 0.0010
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3252 - ce: 1.8891 - loss: 2.5119 - val_accuracy: 0.6000 - val_ce: 1.0001 - val_loss: 1.6185 - learning_rate: 0.0010
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3479 - ce: 1.7046 - loss: 2.3192 - val_accuracy: 0.6500 - val_ce: 1.0128 - val_loss: 1.6231 - learning_rate: 0.0010
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3891 - ce: 1.5269 - loss: 2.1337 - val_accuracy: 0.6333 - val_ce: 0.9837 - val_loss: 1.5865 - learning_rate: 0.0010
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4109 - ce: 1.4831 - loss: 2.0825 - val_accuracy: 0.6000 - val_ce: 0.9660 - val_loss: 1.5616 - learning_rate: 0.0010
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4412 - ce: 1.3797 - loss: 1.97

In [62]:
import pickle
def save_model(model, meta_block, dict_blocks, tsfel_cfg, feature_set_name = "default_tsfel", b_CNN = False):
    feat_global = tsfel_feature_read(dict_blocks, tsfel_cfg, feature_set_name= feature_set_name)
    y_list = []
    xts_list = []

    for i, bid in enumerate(meta_block["block_id"].to_numpy()):
        x_tr_raw, y_i = dict_blocks[int(bid)]
        y_list.append(int(y_i))
        xts_list.append(block_to_windows_postpad(x_tr_raw, T = 2000, stride=2000)[0])

    y_tr = np.array(y_list, dtype=np.int32)

    X_all = feat_global

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_all)
    pickle.dump(scaler, open("models/scaler_features_debug.pkl", "wb"), protocol=5)


    es = tf.keras.callbacks.EarlyStopping(
        monitor="loss", patience=10, restore_best_weights=True
        )
    rlr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss", factor=0.5, patience=max(2, 10 // 2), min_lr=1e-6
        )
    if b_CNN:
        xts_tr = np.stack(xts_list, axis=0).astype(np.float32)
        model.fit({"ts":xts_tr, "feat":X_tr}, y_tr, batch_size=32, epochs=200, callbacks=[es, rlr], shuffle=True)
    else:
        model.fit(X_tr, y_tr,
            callbacks=[es, rlr],
            epochs = 200, shuffle = True, batch_size=32)
    model.save("models/EMG-CNN-debug.keras")

    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter._experimental_lower_tensor_list_ops = False

    tflite_model = converter.convert()

    with open("models/EMG-CNN-debug.tflite", "wb") as f:
        f.write(tflite_model)

In [64]:
cnn_model = build_model(2000, 624)
save_model(cnn_model,meta_blocks, dict_blocks, tsfel_cfg, b_CNN = True)

Epoch 1/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2624 - ce: 6.2222 - loss: 6.8913 - learning_rate: 0.0010
Epoch 2/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3592 - ce: 2.8607 - loss: 3.5247 - learning_rate: 0.0010
Epoch 3/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3952 - ce: 2.0971 - loss: 2.7549 - learning_rate: 0.0010
Epoch 4/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3816 - ce: 1.8817 - loss: 2.5334 - learning_rate: 0.0010
Epoch 5/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4160 - ce: 1.6053 - loss: 2.2510 - learning_rate: 0.0010
Epoch 6/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4520 - ce: 1.5165 - loss: 2.1561 - learning_rate: 0.0010
Epoch 7/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4616 - ce: 1.4296 - loss: 2.0634 - learning_rate: 0.0010
Epoch 8/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4720 - ce: 1.3380 - loss: 1.9662 - learning_rate: 0.0010
Epoch 9/200
40/40 ━━━━━━

INFO:tensorflow:Assets written to: /tmp/tmpafez6vkf/assets


Saved artifact at '/tmp/tmpafez6vkf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 2000, 4), dtype=tf.float32, name='ts'), TensorSpec(shape=(None, 624), dtype=tf.float32, name='feat')]
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  128847115428560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847115434512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847124482000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847124480848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847115428944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847115431632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847309760336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847309764176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847115429136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128847115435088: TensorSpec(sh

W0000 00:00:1788379557.353326  297451 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1788379557.353342  297451 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1788379557.353567  297451 reader.cc:83] Reading SavedModel from: /tmp/tmpafez6vkf
I0000 00:00:1788379557.354117  297451 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1788379557.354123  297451 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpafez6vkf
I0000 00:00:1788379557.358032  297451 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1788379557.358522  297451 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1788379557.383169  297451 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpafez6vkf
I0000 00:00:1788379557.389063  297451 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 35498 microseconds.
I0000 00:00:1788379557.414007  297451